# Validación cruzada ESIOS vs REData (v3 — robusto)

**Objetivo:** Confirmar que las métricas calculadas con REData (datos diarios agregados) cuadran con la realidad medida por ESIOS (datos horarios crudos).

**Periodo de control:** Enero 2024 (mes representativo, sin eventos atípicos).

**Fix v3:** Todas las celdas usan `parse_datetime_safe()` para parsear datetimes con timezones mixtos (problema típico en CSVs REE por cambio horario).

## 1. Configuración y helpers

In [1]:
import requests
import pandas as pd
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURACIÓN
# ============================================================
TOKEN = "e3c302fe80928e3fbf43b229cf573ccf05b9888e65acd3a1394265f73b1e44be"
PROYECTO_DIR = Path(r"C:\Users\Hector\Desktop\premios-steam-2026")
REDATA_DIR = PROYECTO_DIR / "datos" / "redata_raw"
ESIOS_DIR = PROYECTO_DIR / "datos" / "esios_raw" / "validacion"
ESIOS_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "Accept": "application/json; application/vnd.esios-api-v1+json",
    "Content-Type": "application/json",
    "Host": "api.esios.ree.es",
    "x-api-key": TOKEN
}

BASE_URL = "https://api.esios.ree.es"
PERIODO_INICIO = "2024-01-01T00:00"
PERIODO_FIN = "2024-01-31T23:55"

# ============================================================
# HELPER ROBUSTO PARA DATETIMES CON TIMEZONES MIXTOS
# ============================================================
def parse_datetime_safe(serie):
    """
    Parsea una columna datetime que puede tener timezones mixtos
    (típico en CSVs REE: +01:00 en invierno, +02:00 en verano).
    Devuelve datetime en UTC, naive (sin tz info) para evitar comparaciones.
    """
    s = pd.to_datetime(serie, utc=True, errors='coerce')
    return s.dt.tz_localize(None)  # quita info de zona, devuelve naive datetime

def get_esios(indicator_id, start, end, trunc="hour"):
    """Descarga un indicador ESIOS y devuelve DataFrame con datetime ya parseado."""
    r = requests.get(
        f"{BASE_URL}/indicators/{indicator_id}",
        headers=HEADERS,
        params={"start_date": start, "end_date": end, "time_trunc": trunc},
        timeout=60
    )
    r.raise_for_status()
    data = r.json()['indicator']
    df = pd.DataFrame(data['values'])
    if 'datetime' in df.columns:
        df['datetime'] = parse_datetime_safe(df['datetime'])
    return df, data.get('name', '')

print("[OK] Configuración cargada y helpers listos")

[OK] Configuración cargada y helpers listos


## 2. Test de conexión rápido

In [2]:
# Test de que el token funciona
df_test, nombre = get_esios(1739, "2024-01-01T00:00", "2024-01-01T03:00", trunc="hour")
print(f"[OK] Token válido. Indicador test: {nombre}")
print(f"     Filas recibidas: {len(df_test)}")
print(df_test[['datetime', 'value']].head(3))

[OK] Token válido. Indicador test: Precio de la energía excedentaria del autoconsumo para el mecanismo de compensación simplificada (PVPC)
     Filas recibidas: 4
             datetime  value
0 2023-12-31 23:00:00  61.08
1 2024-01-01 00:00:00  48.17
2 2024-01-01 01:00:00  45.34


## 3. VALIDACIÓN — Demanda enero 2024

In [3]:
# === ESIOS — Demanda real horaria ===
df_demanda_esios, nombre = get_esios(1739, PERIODO_INICIO, PERIODO_FIN, trunc="hour")
df_demanda_esios.to_csv(ESIOS_DIR / "demanda_horaria_ene2024.csv", index=False)
demanda_esios_gwh = df_demanda_esios['value'].sum() / 1000
print(f"[ESIOS] {nombre}")
print(f"  Filas horarias: {len(df_demanda_esios)}")
print(f"  Demanda total enero 2024: {demanda_esios_gwh:,.1f} GWh")

# === REData — Demanda diaria ===
redata_demanda = pd.read_csv(REDATA_DIR / "demanda__evolucion.csv")
redata_demanda['datetime'] = parse_datetime_safe(redata_demanda['datetime'])
mask = (
    (redata_demanda['datetime'].dt.year == 2024) &
    (redata_demanda['datetime'].dt.month == 1) &
    (redata_demanda['type'] == 'Demanda')
)
redata_ene2024 = redata_demanda[mask]
demanda_redata_gwh = redata_ene2024['value'].sum() / 1_000_000
print(f"\n[REData] Demanda diaria")
print(f"  Filas: {len(redata_ene2024)}")
print(f"  Demanda total enero 2024: {demanda_redata_gwh:,.1f} GWh")

# === Comparativa ===
diff_pct = abs(demanda_esios_gwh - demanda_redata_gwh) / demanda_esios_gwh * 100
print(f"\n[VALIDACIÓN]")
print(f"  Diferencia: {abs(demanda_esios_gwh - demanda_redata_gwh):,.1f} GWh ({diff_pct:.2f}%)")
if diff_pct < 5:
    print(f"  [OK] COHERENTE — Diferencia <5% es esperable por metodología")
else:
    print(f"  [ALERTA] Diferencia >5% requiere investigación")

[ESIOS] Precio de la energía excedentaria del autoconsumo para el mecanismo de compensación simplificada (PVPC)
  Filas horarias: 744
  Demanda total enero 2024: 53.9 GWh

[REData] Demanda diaria
  Filas: 31
  Demanda total enero 2024: 22.5 GWh

[VALIDACIÓN]
  Diferencia: 31.4 GWh (58.18%)
  [ALERTA] Diferencia >5% requiere investigación


## 4. VALIDACIÓN — Precio mercado diario enero 2024

In [4]:
# === ESIOS — Precio mercado diario ===
df_precio_esios, nombre = get_esios(600, PERIODO_INICIO, PERIODO_FIN, trunc="hour")
df_precio_esios.to_csv(ESIOS_DIR / "precio_mercado_diario_ene2024.csv", index=False)
precio_medio_esios = df_precio_esios['value'].mean()
print(f"[ESIOS] {nombre}")
print(f"  Filas horarias: {len(df_precio_esios)}")
print(f"  Precio medio enero 2024: {precio_medio_esios:.2f} €/MWh")

# === REData — Precio spot ===
redata_precio = pd.read_csv(REDATA_DIR / "mercados__precios-mercados-tiempo-real.csv")
redata_precio['datetime'] = parse_datetime_safe(redata_precio['datetime'])
mask = (
    (redata_precio['datetime'].dt.year == 2024) &
    (redata_precio['datetime'].dt.month == 1) &
    (redata_precio['type'] == 'Precio mercado spot')
)
redata_ene = redata_precio[mask]
# REData devuelve en céntimos×100 según auditoría previa, ESIOS en €/MWh directos
precio_medio_redata = redata_ene['value'].mean() / 100
print(f"\n[REData] Precio spot")
print(f"  Filas: {len(redata_ene)}")
print(f"  Precio medio enero 2024: {precio_medio_redata:.2f} €/MWh")

# === Comparativa ===
diff_pct = abs(precio_medio_esios - precio_medio_redata) / precio_medio_esios * 100
print(f"\n[VALIDACIÓN]")
print(f"  Diferencia: {abs(precio_medio_esios - precio_medio_redata):.2f} €/MWh ({diff_pct:.2f}%)")
if diff_pct < 10:
    print(f"  [OK] COHERENTE")
else:
    print(f"  [ALERTA] REData puede estar reportando un producto distinto al diario OMIE")
    print(f"           Esto es lo que detectó la auditoría senior. Documentar en memoria.")

[ESIOS] Precio mercado SPOT Diario
  Filas horarias: 4464
  Precio medio enero 2024: 76.38 €/MWh

[REData] Precio spot
  Filas: 744
  Precio medio enero 2024: 0.74 €/MWh

[VALIDACIÓN]
  Diferencia: 75.64 €/MWh (99.03%)
  [ALERTA] REData puede estar reportando un producto distinto al diario OMIE
           Esto es lo que detectó la auditoría senior. Documentar en memoria.


## 5. VALIDACIÓN — Renovable enero 2024

In [6]:
# === ESIOS — Renovables ===
tecnologias_renovables = {
    549: "Hidráulica",
    551: "Eólica",
    552: "Solar fotovoltaica",
    553: "Solar térmica"
}

total_renovable_esios = 0
for ind_id, nombre in tecnologias_renovables.items():
    df, _ = get_esios(ind_id, PERIODO_INICIO, PERIODO_FIN, trunc="hour")
    
    # PROTECCIÓN: si no hay columna 'value', saltar al siguiente
    if df.empty or 'value' not in df.columns:
        print(f"  [WARN] ESIOS {nombre} (ID {ind_id}): sin datos disponibles")
        continue
    
    suma_gwh = df['value'].sum() / 1000
    print(f"  ESIOS {nombre} (ID {ind_id}): {suma_gwh:,.1f} GWh")
    total_renovable_esios += suma_gwh
    time.sleep(0.5)

print(f"\n[ESIOS] Renovable total enero 2024: {total_renovable_esios:,.1f} GWh")

  ESIOS Hidráulica (ID 549): 61,828.2 GWh
  ESIOS Eólica (ID 551): 67,779.1 GWh
  [WARN] ESIOS Solar fotovoltaica (ID 552): sin datos disponibles
  ESIOS Solar térmica (ID 553): -6,016.9 GWh

[ESIOS] Renovable total enero 2024: 123,590.5 GWh


## 6. RESUMEN DE VALIDACIÓN

Si todas las validaciones devuelven diferencias <5-10%, el modelo REData actual es estadísticamente coherente y se puede defender en la memoria con rigor.

Documenta los resultados en la memoria como sección de **"Validación cruzada metodológica"** — eso impresiona al jurado porque demuestra rigor científico profesional.